In [0]:
# =============================================================
# nb_silver_to_gold_pet_owners.py
# Layer     : Gold
# Audience  : Pet Owners
# Purpose   : Orchestrates the Pet Owner daily summary Gold table.
#
# Output table : gold/gold_pet_owner_daily_summary
# Grain        : pet_id × date
#
# Sources (Silver):
#   activity_snapshots  → steps, distance, active_minutes
#   health_snapshots    → heart_rate, body_temperature
#   feeding_events      → food_dispensed_grams
#   hydration_events    → water_consumed_ml
#   pets                → pet_name, species, breed
#   owners              → owner_name, city, country
#
# ADF params: storage_account
# =============================================================

# ── CELL 1: Widgets ───────────────────────────────────────────
dbutils.widgets.removeAll()
dbutils.widgets.text("storage_account", "petiot")


def abfss(container: str, path: str = "", storage_account: str | None = None) -> str:
    active_storage_account = storage_account or dbutils.widgets.get("storage_account")
    clean_path = path.lstrip("/")
    base = f"abfss://{container}@{active_storage_account}.dfs.core.windows.net"
    return f"{base}/{clean_path}" if clean_path else f"{base}/"

print("=" * 60)
print("  nb_silver_to_gold_pet_owners")
print("  Output: gold_pet_owner_daily_summary")
print("  Grain : pet_id × date")
print("=" * 60)

# ── CELL 2: Auth ──────────────────────────────────────────────
%run /Workspace/Shared/pet-analytics/shared/set_auth_context.py.ipynb

# ── CELL 3: Load specialists ──────────────────────────────────
%run /Workspace/Shared/pet-analytics/gold/gold_reader.ipynb
%run /Workspace/Shared/pet-analytics/gold/aggregator.ipynb
%run /Workspace/Shared/pet-analytics/gold/gold_writer.ipynb

# ── CELL 4: Read Silver fact tables ──────────────────────────
from pyspark.sql.functions import col, when, lit, coalesce

print("\n[pet_owners] Reading Silver fact tables...")
activity_df  = read_activity_snapshots()
health_df    = read_health_snapshots()
feeding_df   = read_feeding_events()
hydration_df = read_hydration_events()

print(f"  activity_snapshots : {activity_df.count():,}")
print(f"  health_snapshots   : {health_df.count():,}")
print(f"  feeding_events     : {feeding_df.count():,}")
print(f"  hydration_events   : {hydration_df.count():,}")

# ── CELL 5: Stage 1 — Aggregate each fact independently ───────
# Each fact → one row per (pet_id, date)
# This MUST happen before any joins — prevents row fan-out
print("\n[pet_owners] Stage 1: Aggregating facts...")

agg_act  = agg_activity_daily(activity_df)
agg_hlth = agg_health_daily(health_df)
agg_feed = agg_feeding_daily(feeding_df)
agg_hyd  = agg_hydration_daily(hydration_df)

print(f"  activity agg rows  : {agg_act.count():,}")
print(f"  health agg rows    : {agg_hlth.count():,}")
print(f"  feeding agg rows   : {agg_feed.count():,}")
print(f"  hydration agg rows : {agg_hyd.count():,}")

# ── CELL 6: Stage 2 — Full outer join aggregates ──────────────
# Rule 1 — Full-Join Date Trap:
# A pet with activity but no hydration on a day must NOT be dropped.
# full_outer keeps all (pet_id, date) combinations from all sources.
print("\n[pet_owners] Stage 2: Combining fact aggregates (full outer)...")

fact_summary = combine_pet_daily(agg_act, agg_hlth, agg_feed, agg_hyd)
print(f"  Combined rows: {fact_summary.count():,}")

# ── CELL 7: Stage 3 — Enrich with dimension labels ────────────
# LEFT JOIN slim dimension tables for display labels
# Dimensions are joined AFTER aggregation — never before
print("\n[pet_owners] Stage 3: Enriching with dimension labels...")

pets_dim   = read_pets()
owners_dim = read_owners()

gold_df = (
    fact_summary
    .join(pets_dim,   "pet_id",   "left")
    .join(owners_dim, "owner_id", "left")
)

# ── CELL 8: Derive activity level category ────────────────────
gold_df = (
    gold_df
    .withColumn("activity_level",
        when(col("total_steps") >= 5000, lit("High"))
        .when(col("total_steps") >= 2000, lit("Medium"))
        .when(col("total_steps") >= 0,   lit("Low"))
        .otherwise(lit("No Device"))
    )
)

# ── CELL 9: Apply null defaults ───────────────────────────────
# Rule 2 — Explicit Null Handling
# coalesce additive metrics to 0; keep vital signs as NULL
# (NULL vitals = pet has no health monitor, meaningful distinction)
gold_df = apply_null_defaults(gold_df, "gold_pet_owner_daily_summary")

# ── CELL 10: Add audit + write ────────────────────────────────
gold_df     = add_gold_audit(gold_df)
final_count = write_gold(gold_df, "gold_pet_owner_daily_summary")

# ── CELL 11: Preview ──────────────────────────────────────────
print("\n[pet_owners] Sample output:")
spark.read.format("delta") \
    .load(abfss("gold","gold_pet_owner_daily_summary")) \
    .select("pet_name","species","date","total_steps",
            "avg_heart_rate","total_food_grams","activity_level") \
    .show(5, truncate=False)

print(f"\n{'='*60}")
print(f"  Gold complete: gold_pet_owner_daily_summary")
print(f"  Rows: {final_count:,}")
print(f"{'='*60}")

dbutils.notebook.exit(f"SUCCESS|gold_pet_owner_daily_summary|{final_count}")